In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [ ]:
people_from_db_file = Path(project_root / "data/from db/people_db.json")

with open(people_from_db_file, "r") as f:
   new_id = json.load(f)

# rprint(new_id[:2])

new_uid_dict = {u["unified_id"]: u["person_id"] for u in new_id}
# rprint(dict(list(new_uid_dict.items())[:3]))


In [3]:
b2p_matched_data_file = Path(project_root / "data/people/prepping4loading/b2p_matched_data.json")
with open(b2p_matched_data_file, "r") as f:
   matched = json.load(f)

matched_uid_dict = {
   v["unified_id"]: {
      "person_id": None,
      "person_id_old": v["person_id_old"],
      "entries": v["entries"]
   } for k, v in matched.items()}

# rprint(dict(list(matched_uid_dict.items())[:2]))


In [4]:
b2p_remapped_data = {}
b2p_remapped_list = []
seen_combinations = set()
update_count = 0
problems = {}

roles = [
    "is_author",
    "is_editor",
    "is_contributor",
    "is_translator"
]
updates = {}
problem_count = 0

for uid, person in matched_uid_dict.items():
# for uid, person in list(matched_uid_dict.items())[:20]:
    if not uid in new_uid_dict:
        problem_count += 1
        problems[uid] = person
        continue
    id_old = person["person_id_old"]
    person_id = new_uid_dict[uid]
    entries = person["entries"]

    for entry in entries:
        composite_id = entry["composite_id"]
        combo = (uid, composite_id)

        if combo not in seen_combinations:
            seen_combinations.add(combo)
            b2p_remapped_list.append({
              "unified_id": uid,
              **entry,
              "person_id": person_id,
              "person_id_old": id_old
           })
        else:
            existing = [e for e in b2p_remapped_list if e["person_id"] == person_id and e["composite_id"] == composite_id][0]

            existing.update({
               role: existing[role] or entry[role] for role in roles
            })
            update_count += 1

            updates[uid] = {
               "count": update_count,
               "entry_old": entry,
               "existing_updated": existing
            }
        b2p_remapped_data[person_id] = {
           "person_id": person_id,
           "person_id_old": id_old,
           "unified_id": uid,
           "entries": entries
        }

prep_folder_path = Path(project_root / "data/people/prepping4loading")

b2p_remapped_data_file = Path(project_root / prep_folder_path / "b2p_remapped_data.json")
b2p_remapped_list_file = Path(project_root / prep_folder_path / "b2p_remapped_list.json")

with open(b2p_remapped_data_file, "w") as f:
    json.dump(b2p_remapped_data, f, ensure_ascii=False, indent=2)
with open(b2p_remapped_list_file, "w") as f:
    json.dump(b2p_remapped_list, f, ensure_ascii=False, indent=2)

rprint(f"""
=== LOG ===
len list: {len(b2p_remapped_list)}
len data: {len(b2p_remapped_data)}
problems: {problem_count}

=== DONE ===
""")


# rprint(problem_count)
# with open("problems.json", "w") as f:
#     json.dump(problems, f, ensure_ascii=False, indent=2)


=== LOG ===
len list: 9234
len data: 5103
problems: 0

=== DONE ===